# Student Performance Prediction System

**Objective:** Predict student academic performance using attendance, assignment scores, internal assessments, and other academic indicators.

**ML Concepts:** Data Cleaning · Feature Engineering · Regression Models · Model Evaluation

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import clean_student_data
from src.evaluation import compare_models, evaluate_model
from src.feature_engineering import engineer_features, get_feature_columns, build_preprocessor
from src.model_training import get_models, build_pipeline

sns.set_style('whitegrid')
print(f'Project root: {PROJECT_ROOT}')

## 1. Load & Explore Data

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'student_performance.csv'

if not data_path.exists():
    from generate_data import generate_student_data
    data_path.parent.mkdir(parents=True, exist_ok=True)
    generate_student_data().to_csv(data_path, index=False)

raw_df = pd.read_csv(data_path)
print(f'Shape: {raw_df.shape}')
raw_df.head()

In [ ]:
raw_df.info()
print('\nMissing values:')
print(raw_df.isnull().sum())
print('\nDuplicates:', raw_df.duplicated(subset=['student_id']).sum())

In [ ]:
raw_df.describe()

## 2. Data Cleaning

- Remove duplicate records
- Clip scores to valid ranges
- Handle outliers using IQR method
- Impute missing values (median for numeric, mode for categorical)

In [ ]:
cleaned_df = clean_student_data(raw_df)
print(f'Records: {len(raw_df)} -> {len(cleaned_df)}')
print(f'Missing values after cleaning: {cleaned_df.isnull().sum().sum()}')

## 3. Feature Engineering

Create derived features that capture academic progress and engagement patterns.

In [ ]:
featured_df = engineer_features(cleaned_df)
feature_cols = get_feature_columns()

featured_df[feature_cols + ['final_score']].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(featured_df['final_score'], kde=True, ax=axes[0])
axes[0].set_title('Distribution of Final Scores')

corr_cols = ['attendance_pct', 'assignment_avg', 'academic_progress', 'engagement_score', 'final_score']
sns.heatmap(featured_df[corr_cols].corr(), annot=True, cmap='coolwarm', ax=axes[1])
axes[1].set_title('Feature Correlations')
plt.tight_layout()
plt.show()

## 4. Train Regression Models

Compare Linear Regression, Ridge, Random Forest, and Gradient Boosting.

In [ ]:
X = featured_df[feature_cols]
y = featured_df['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
preprocessor = build_preprocessor(feature_cols)
results = []
trained_pipelines = {}

for name, model in get_models().items():
    pipeline = build_pipeline(preprocessor, model)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results.append(evaluate_model(y_test, y_pred, name))
    trained_pipelines[name] = pipeline

comparison = compare_models(results)
comparison

## 5. Model Evaluation

Compare models using **MAE**, **RMSE**, and **R²** metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=comparison, x='model', y='R2', ax=axes[0], palette='viridis')
axes[0].set_title('R² Score (Higher is Better)')
axes[0].tick_params(axis='x', rotation=25)

sns.barplot(data=comparison, x='model', y='RMSE', ax=axes[1], palette='magma')
axes[1].set_title('RMSE (Lower is Better)')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
best_name = comparison.iloc[0]['model']
best_pipeline = trained_pipelines[best_name]
y_pred_best = best_pipeline.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_best, alpha=0.6, edgecolors='k', linewidth=0.5)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax.set_xlabel('Actual Final Score')
ax.set_ylabel('Predicted Final Score')
ax.set_title(f'Actual vs Predicted ({best_name})')
plt.show()

## 6. Sample Predictions

In [ ]:
sample = X_test.iloc[:5].copy()
sample['actual'] = y_test.iloc[:5].values
sample['predicted'] = best_pipeline.predict(sample[feature_cols])
sample['error'] = (sample['predicted'] - sample['actual']).round(2)
sample[['attendance_pct', 'assignment_avg', 'midterm_score', 'actual', 'predicted', 'error']]